# Pipeline completo: INGESTA → DINO → SAM3

Notebook único que recorre el pipeline de investigación de punta a punta usando
los videos ya normalizados de `dataset/processed/` (los que importan para el
proyecto):

1. **Ingesta** — elige un video de `dataset/processed/`, muestra su metadata real
   (frames, fps, resolución, duración) y, si `RUN_INGEST=True`, transcodifica
   primero los crudos nuevos de `dataset/unprocessed/` con `process_video_dataset`.
2. **Enhance (opcional)** — `process_marine_video_ffmpeg` limpia el video (inpaint
   de glints, suavizado preservando bordes, CLAHE); DINO puede leer la versión
   mejorada con `DINO_SOURCE="enhanced"`.
3. **DINO** — `run_zoom_detector` detecta anomalías coarse-to-fine sobre los
   keyframes del video de trabajo y devuelve puntos normalizados listos para SAM3.
   Hay diagnóstico del frame elegido (heatmap, máscara, árbol de zoom) y resumen
   por keyframe.
4. **SAM3** — cada punto DINO se convierte en un point prompt y SAM3 rastrea los
   masklets por todo el video. Modos: `chunked` (sesiones con solape, VRAM
   acotada), `incremental` (una sola sesión) y `simple` (prompts en un frame,
   único modo que admite `PROMPT_MODE="text"`, p. ej. `"dolphin"`).
5. **Verificación** — checklist PASS/FALLO por etapa, estadísticas de tracks
   (objetos por frame, área por objeto) y video overlay con las máscaras.

Los interruptores de la primera celda (`RUN_INGEST`, `RUN_ENHANCE`, `RUN_DINO`,
`RUN_SAM`, `SAM_MODE`, `PROMPT_MODE`) permiten ejecutar solo las etapas que
interesen; `START_S`/`DURATION_S` recortan el video para corridas de prueba.

**Requisitos**: `HF_TOKEN` en `.env` (checkpoints gated de SAM3/DINOv3),
`ffmpeg`/`ffprobe` + `exiftool` en el sistema, y una GPU de ~24 GB (RTX 4090). Los
videos de `dataset/processed/` los genera `01_ingest.ipynb`.

## 0. Configuración

Todos los parámetros del pipeline viven en esta celda. `AnomalyOptions` agrupa
los modos de score y máscara de DINO; el resto son las perillas de cada etapa.

In [ ]:
# Debe fijarse ANTES de la primera reserva de CUDA: el estado de SAM3 crece con
# objetos x frames y la fragmentación del allocator produce OOM en corridas largas.
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

%matplotlib inline
import json

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

from fish_segmentation.dinov3 import AnomalyOptions
from fish_segmentation.enhance import process_marine_video_ffmpeg
from fish_segmentation.ingest import process_video, process_video_dataset
from fish_segmentation.paths import find_repo_root, load_env
from fish_segmentation.video_io import (
    load_all_frames_rgb,
    load_frame_rgb,
    load_sampled_frames,
)

load_env()
ROOT = find_repo_root()
PROCESSED_DIR = ROOT / "dataset" / "processed"
UNPROCESSED_DIR = ROOT / "dataset" / "unprocessed"
OUTPUTS = ROOT / "notebooks" / "outputs"
OUTPUTS.mkdir(parents=True, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")

# ============================ ETAPAS ============================
DATASET_ID = "20260813_213529_7f994a2f"  # carpeta de dataset/processed/ (None -> la más reciente)
START_S = 0.0          # segundo inicial del recorte de trabajo (0.0 = desde el principio)
DURATION_S = None      # duración del recorte en segundos (None = video completo)
RUN_INGEST = False     # transcodificar dataset/unprocessed/ -> dataset/processed/
RUN_ENHANCE = False    # generar <stem>_enhanced.mp4 (glints + suavizado + CLAHE)
DINO_SOURCE = "processed"  # "processed" | "enhanced": de dónde lee DINO los frames
RUN_DINO = True
RUN_SAM = True

# ============================ INGESTA ============================
INGEST_FPS = 15       # fps de normalización del dataset (todos los procesados están a 15)
INGEST_WIDTH = None   # None = mantener la resolución original (4K en el dataset)
INGEST_HEIGHT = None
INGEST_CRF = 18

# ============================ ENHANCE ============================
ENHANCE_GLINT_THRESHOLD = 235
ENHANCE_CLAHE_CLIP = 2.5
ENHANCE_CLAHE_TILE = (8, 8)

# ============================ DINO ============================
DINO_LONG_SIDE = 1024         # lado largo máximo de cada vista de la recursión
DINO_RESOLUTION_SCALE = 4.0   # multiplicador de resolución relativa (ajustado al 4K)
DINO_PERCENTILE = 99.5
DINO_MAX_PATCHES = 4096
DINO_MAX_LEVELS = 4
ZONE_MERGE_GAP = 0.08         # fusiona cajas cercanas en una zona de contexto
MIN_ZONE_FRACTION = 0.25      # tamaño mínimo de zona (fracción del lado largo)
MIN_CONFIRMATIONS = 2         # solo detecciones confirmadas en >= 2 niveles de zoom
KEYFRAME_STRIDE = 30          # DINO y SAM corren cada 30 frames (2 s @ 15 fps)
ANOMALY_OPTIONS = AnomalyOptions(
    score_mode="mean",           # "mean" (validado) | "knn" (ver 08_dinov3_heatmap_ab)
    knn_k=5,
    local_contrast_strength=0.0,
    local_contrast_sigma_pct=0.05,
    mask_method="hysteresis",    # "hysteresis" (validado) | "adaptive"
    low_percentile=98.0,
    closing_kernel_size=0,
)

# ============================ SAM3 ============================
SAM_MODE = "chunked"         # "chunked" | "incremental" | "simple"
PROMPT_MODE = "dino_points"  # "dino_points" | "text" (texto solo en SAM_MODE="simple")
TEXT_PROMPT = "dolphin"
SESSION_FPS = 15             # fps del proxy de SAM3 (el dataset ya está a 15)
PROXY_WIDTH = 1024
PROXY_HEIGHT = 576
MAX_OBJECTS = 24             # tope global de masklets
MIN_SCORE = 0.5              # filtros de calidad para candidatos nuevos
MIN_SOLIDITY = 0.25
MIN_AREA = None              # 'area' está en píxeles de la vista: no comparable entre niveles
MASK_MARGIN_PX = 4
DEDUPE_OVERLAP = 0.3
OFFLOAD_STATE_TO_CPU = True  # estado del tracker (objetos x frames) -> RAM de CPU
OFFLOAD_VIDEO_TO_CPU = True

# Chunked: 150 frames acotan el estado por sesión; 30 frames solapan y arrastran ids.
CHUNK_FRAMES = 150
OVERLAP_FRAMES = 30
CARRY_MIN_AREA = 16

# ============================ DIAGNÓSTICO ============================
SHOW_DINO_DIAGNOSTIC = True  # heatmap + máscara + árbol de zoom del frame elegido
DIAGNOSTIC_KEYFRAME = 0      # keyframe del diagnóstico detallado de DINO
SHOW_KEYFRAME_REVIEW = True  # figura por keyframe de SAM (máscaras vs candidatos)
N_OVERVIEW_FRAMES = 3        # keyframes mostrados en el resumen visual de DINO
print(
    f"etapas: ingest={RUN_INGEST} enhance={RUN_ENHANCE} "
    f"dino={RUN_DINO} sam={RUN_SAM} | SAM_MODE={SAM_MODE} PROMPT_MODE={PROMPT_MODE}"
)

## 1. INGESTA — elegir y validar el video de `dataset/processed/`

La ingesta convierte los crudos DJI de `dataset/unprocessed/` en
`dataset/processed/<timestamp>_<hash>/video.mp4` + `metadata.json` con
`fish_segmentation.ingest.process_video_dataset` (normaliza a 15 fps; la
resolución original 4K se mantiene).

Esta sección:

- lista **todos** los videos procesados con su metadata real (leída del archivo,
  no del sidecar: el sidecar describe el crudo y por eso reporta 60 fps);
- selecciona el de `DATASET_ID` (`None` = el más reciente);
- con `RUN_INGEST=True`, transcodifica primero los crudos nuevos;
- recorta `START_S`/`DURATION_S` a un video de trabajo en `notebooks/outputs/`
  para corridas cortas de prueba;
- verifica frames/fps/duración y muestra una muestra de frames.

La columna "grabación" viene del sidecar y sirve para reconocer vuelo/fecha.

In [ ]:
# Metadata real del video normalizado (cv2), no la del crudo original.
processed_folders = sorted(PROCESSED_DIR.glob("*")) if PROCESSED_DIR.is_dir() else []
processed_entries = []
for folder in processed_folders:
    video_file = folder / "video.mp4"
    if not video_file.is_file():
        continue
    metadata_path = folder / "metadata.json"
    metadata = (
        json.loads(metadata_path.read_text(encoding="utf-8"))
        if metadata_path.is_file()
        else {}
    )
    capture = cv2.VideoCapture(str(video_file))
    file_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    file_fps = capture.get(cv2.CAP_PROP_FPS)
    file_width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    file_height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    capture.release()
    processed_entries.append(
        {
            "dataset_id": folder.name,
            "path": video_file,
            "recording": metadata.get("recording_timestamp", "?"),
            "frames": file_frames,
            "fps": file_fps,
            "width": file_width,
            "height": file_height,
            "duration_s": file_frames / file_fps if file_fps else float("nan"),
        }
    )

print(f"{len(processed_entries)} videos en dataset/processed/:")
print(f"{'dataset_id':<26} {'grabación':<16} {'frames':>7} {'fps':>6} {'resolución':>12} {'duración':>9}")
for entry in processed_entries:
    print(
        f"{entry['dataset_id']:<26} {entry['recording']:<16} {entry['frames']:>7} "
        f"{entry['fps']:>6.2f} {entry['width']:>5}x{entry['height']:<5} "
        f"{entry['duration_s']:>8.1f}s"
    )

selected_entry = (
    processed_entries[-1]
    if DATASET_ID is None and processed_entries
    else next((e for e in processed_entries if e["dataset_id"] == DATASET_ID), None)
)
assert selected_entry is not None, (
    f"no hay video para DATASET_ID={DATASET_ID!r} en {PROCESSED_DIR}; "
    f"disponibles: {[e['dataset_id'] for e in processed_entries]}"
)
PROCESSED_VIDEO = selected_entry["path"]
print(f"video seleccionado: {PROCESSED_VIDEO.relative_to(ROOT)}")

In [ ]:
if RUN_INGEST:
    process_video_dataset(str(UNPROCESSED_DIR), str(PROCESSED_DIR))
    print("Ingesta terminada: vuelve a ejecutar la celda anterior para refrescar la tabla.")
else:
    print("RUN_INGEST=False: se usa directamente el video normalizado de dataset/processed/.")

In [ ]:
# Recorte de trabajo: mantiene cortas las corridas de prueba. Sin recorte se usa
# el video completo de dataset/processed/.
stem = PROCESSED_VIDEO.parent.name
if START_S > 0 or DURATION_S is not None:
    clip_suffix = f"_clip_{START_S:.0f}s"
    if DURATION_S is not None:
        clip_suffix += f"_{DURATION_S:.0f}s"
    working_video = OUTPUTS / f"{stem}{clip_suffix}.mp4"
    if not working_video.exists():
        process_video(
            input_path=str(PROCESSED_VIDEO),
            output_path=str(working_video),
            start_time=START_S,
            duration=DURATION_S,
            fps=INGEST_FPS,
            crf=INGEST_CRF,
        )
else:
    working_video = PROCESSED_VIDEO

capture = cv2.VideoCapture(str(working_video))
n_work_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
work_fps = capture.get(cv2.CAP_PROP_FPS)
work_width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
work_height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
capture.release()
work_duration = n_work_frames / work_fps if work_fps else float("nan")
print(f"video de trabajo: {working_video.relative_to(ROOT)}")
print(f"  {n_work_frames} frames @ {work_fps:.2f} fps, {work_width}x{work_height}, {work_duration:.1f}s")

ingest_checks = {
    "video de trabajo existe": working_video.is_file(),
    "frames > 0": n_work_frames > 0,
    "fps normalizado": abs(work_fps - INGEST_FPS) < 0.5,
    "duración > 0": work_duration > 0,
}
for name, passed in ingest_checks.items():
    print(f"  [{'OK' if passed else 'FALLO'}] {name}")

# --- Verificación visual: muestra de la ingesta ---
sample_frames = load_sampled_frames(str(working_video), 3)
fig, axes = plt.subplots(1, len(sample_frames), figsize=(15, 4), squeeze=False)
for ax, frame in zip(axes[0], sample_frames):
    ax.imshow(frame)
    ax.axis("off")
fig.suptitle(
    f"Ingesta: muestra de {working_video.name} ({n_work_frames} frames @ {work_fps:.1f} fps)"
)
plt.tight_layout()
plt.show()

## 2. ENHANCE (opcional) — limpieza marina

`process_marine_video_ffmpeg` aplica el pipeline de `enhance.py` frame a frame:
inpaint de glints/espuma (brillo > `ENHANCE_GLINT_THRESHOLD` en HSV), suavizado
preservando bordes y CLAHE sobre el canal L de LAB. Con `RUN_ENHANCE=True` se
escribe `notebooks/outputs/<stem>_enhanced.mp4`.

`DINO_SOURCE` decide qué lee DINO: `"processed"` (el video de trabajo tal cual,
por defecto) o `"enhanced"` (el mejorado). Ojo: si el suavizado borra el patrón
fino de las aletas, el heatmap de DINO pierde señal; en ese caso vuelve a
`DINO_SOURCE="processed"`.

La verificación muestra original vs mejorado y el mapa de diferencia.

In [ ]:
if RUN_ENHANCE:
    enhanced_video = OUTPUTS / f"{working_video.stem}_enhanced.mp4"
    if not enhanced_video.exists():
        process_marine_video_ffmpeg(
            input_video_path=str(working_video),
            output_video_path=str(enhanced_video),
            fps=SESSION_FPS,
            glint_threshold=ENHANCE_GLINT_THRESHOLD,
            clahe_clip_limit=ENHANCE_CLAHE_CLIP,
            clahe_tile_grid_size=ENHANCE_CLAHE_TILE,
        )
    print(f"video mejorado: {enhanced_video.relative_to(ROOT)}")

    # --- Verificación visual: antes / después ---
    probe_index = min(n_work_frames // 2, max(0, n_work_frames - 1))
    before = load_frame_rgb(str(working_video), probe_index)
    after = load_frame_rgb(str(enhanced_video), probe_index)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].imshow(before)
    axes[0].set_title(f"original (frame {probe_index})")
    axes[0].axis("off")
    axes[1].imshow(after)
    axes[1].set_title("mejorado (deglint + suavizado + CLAHE)")
    axes[1].axis("off")
    if after is not None and before is not None and after.shape == before.shape:
        difference = cv2.absdiff(
            cv2.cvtColor(before, cv2.COLOR_RGB2BGR),
            cv2.cvtColor(after, cv2.COLOR_RGB2BGR),
        )
        axes[2].imshow(difference.sum(axis=2), cmap="magma")
        axes[2].set_title(f"|diferencia| (media {difference.mean():.2f})")
        axes[2].axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("RUN_ENHANCE=False: no se genera la versión mejorada.")

DINO_VIDEO = enhanced_video if (RUN_ENHANCE and DINO_SOURCE == "enhanced") else working_video
if DINO_SOURCE == "enhanced" and not RUN_ENHANCE:
    print("AVISO: DINO_SOURCE='enhanced' con RUN_ENHANCE=False -> se usa el video de trabajo.")
print(f"DINO leerá: {DINO_VIDEO.relative_to(ROOT)}")

## 3. DINO — detección coarse-to-fine de anomalías

`run_zoom_detector` parte de una vista reducida (lado largo ≤ `DINO_LONG_SIDE`),
calcula un heatmap de anomalías comparando cada parche ViT contra el resto
(`AnomalyOptions`: similitud media o kNN, contraste local, máscara por
histéresis) y recursa en las zonas de interés hasta agotar `DINO_MAX_LEVELS` o
llegar a resolución nativa. `DINO_RESOLUTION_SCALE=4.0` está ajustado al 4K del
dataset y el merge "deepest-wins" devuelve puntos normalizados a la resolución
original, listos para SAM3.

- El frame `DIAGNOSTIC_KEYFRAME` se inspecciona a fondo: heatmap + máscara cruda,
  árbol de zoom y detalle de la vista raíz (input / heatmap / máscara / puntos).
- Después DINO corre sobre todos los keyframes (cada `KEYFRAME_STRIDE` frames) y
  la caché JSON resultante es la que consume SAM3 en la sección siguiente.
- Los plots de cierre resumen candidatos por keyframe, `score_mean` y
  confirmaciones (`n_levels`), más una vista de keyframes de muestra.

In [ ]:
from fish_segmentation.dinov3 import (
    load_backbone,
    plot_detection_results,
    plot_zoom_level,
    plot_zoom_overview,
    run_dino_view,
    run_zoom_detector,
)

model, processor = load_backbone("facebook/dinov3-vitl16-pretrain-lvd1689m", device=device)
print("DINOv3 cargado: facebook/dinov3-vitl16-pretrain-lvd1689m")

In [ ]:
if RUN_DINO and SHOW_DINO_DIAGNOSTIC:
    diag_index = min(DIAGNOSTIC_KEYFRAME, max(0, n_work_frames - 1))
    diagnostic_image = Image.fromarray(load_frame_rgb(str(DINO_VIDEO), diag_index))
    trace = []
    diag_detections = run_zoom_detector(
        image=diagnostic_image,
        model=model,
        processor=processor,
        long_side=DINO_LONG_SIDE,
        resolution_scale=DINO_RESOLUTION_SCALE,
        percentile_threshold=DINO_PERCENTILE,
        max_levels=DINO_MAX_LEVELS,
        max_patches=DINO_MAX_PATCHES,
        merge_gap=ZONE_MERGE_GAP,
        min_zone_fraction=MIN_ZONE_FRACTION,
        min_confirmations=MIN_CONFIRMATIONS,
        options=ANOMALY_OPTIONS,
        trace=trace,
        device=device,
    )
    print(f"frame {diag_index}: {len(diag_detections)} detecciones, {len(trace)} vistas DINO")
    for index, det in enumerate(diag_detections):
        print(
            f"  obj {index}: L{det['level']} n_levels={det['n_levels']} "
            f"xy=({det['x']:.3f}, {det['y']:.3f}) r={det['radius']:.4f} "
            f"score={det['score_mean']:.3f} solidity={det['solidity']:.2f} "
            f"bbox={tuple(round(v, 3) for v in det['bbox'])}"
        )

    # (a) señal cruda de la vista coarse: imagen / heatmap / máscara umbralizada
    coarse_mask, coarse_heatmap, cropped_frame = run_dino_view(
        image=diagnostic_image,
        model=model,
        processor=processor,
        long_side=DINO_LONG_SIDE,
        resolution_scale=DINO_RESOLUTION_SCALE,
        percentile_threshold=DINO_PERCENTILE,
        max_patches=DINO_MAX_PATCHES,
        options=ANOMALY_OPTIONS,
        device=device,
    )
    plot_detection_results(cropped_frame, coarse_heatmap, coarse_mask)

    # (b) árbol de zoom completo + detecciones finales del merge
    plot_zoom_overview(diagnostic_image, trace, diag_detections)

    # (c) detalle de la primera vista (raíz): input / heatmap / máscara / puntos
    if trace:
        plot_zoom_level(diagnostic_image, trace[0])
else:
    print("RUN_DINO=False o SHOW_DINO_DIAGNOSTIC=False: sin diagnóstico detallado.")

In [ ]:
dino_detections_cache = {}
dino_keyframes = list(range(0, n_work_frames, KEYFRAME_STRIDE))

if RUN_DINO:
    if not dino_keyframes:
        print("AVISO: el video de trabajo no tiene frames.")
    torch.cuda.reset_peak_memory_stats() if device == "cuda" else None
    for k in dino_keyframes:
        keyframe = load_frame_rgb(str(DINO_VIDEO), k)
        if keyframe is None:
            print(f"keyframe {k:5d}: frame no legible, se omite")
            continue
        detections = run_zoom_detector(
            image=Image.fromarray(keyframe),
            model=model,
            processor=processor,
            long_side=DINO_LONG_SIDE,
            resolution_scale=DINO_RESOLUTION_SCALE,
            percentile_threshold=DINO_PERCENTILE,
            max_levels=DINO_MAX_LEVELS,
            max_patches=DINO_MAX_PATCHES,
            merge_gap=ZONE_MERGE_GAP,
            min_zone_fraction=MIN_ZONE_FRACTION,
            min_confirmations=MIN_CONFIRMATIONS,
            options=ANOMALY_OPTIONS,
            device=device,
        )
        dino_detections_cache[k] = detections
        print(f"keyframe {k:5d}: {len(detections):2d} candidatos")

    dino_cache_path = OUTPUTS / f"{working_video.stem}_dino_detections.json"
    with open(dino_cache_path, "w", encoding="utf-8") as handle:
        json.dump({str(k): v for k, v in dino_detections_cache.items()}, handle, indent=2)
    print(f"caché DINO: {dino_cache_path.relative_to(ROOT)}")
else:
    dino_cache_path = OUTPUTS / f"{working_video.stem}_dino_detections.json"
    if dino_cache_path.is_file():
        with open(dino_cache_path, encoding="utf-8") as handle:
            dino_detections_cache = {int(k): v for k, v in json.load(handle).items()}
        print(
            f"RUN_DINO=False: se reutiliza la caché {dino_cache_path.name} "
            f"({len(dino_detections_cache)} keyframes)"
        )
    else:
        print("RUN_DINO=False y no hay caché DINO: SAM3 no tendrá candidatos.")

In [ ]:
if RUN_DINO and any(len(dets) for dets in dino_detections_cache.values()):
    counts = [len(dino_detections_cache.get(k, [])) for k in dino_keyframes]
    all_dets = [det for dets in dino_detections_cache.values() for det in dets]
    scores = [det["score_mean"] for det in all_dets]
    confirmations = [det["n_levels"] for det in all_dets]

    fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
    axes[0].bar(dino_keyframes, counts, color="tab:blue")
    axes[0].set_title("candidatos DINO por keyframe")
    axes[0].set_xlabel("frame")
    axes[0].set_ylabel("nº detecciones")
    if scores:
        axes[1].hist(scores, bins=20, color="tab:orange")
        axes[1].set_title(f"score_mean de los candidatos (n={len(scores)})")
        axes[1].set_xlabel("score_mean")
    axes[2].hist(
        confirmations,
        bins=range(1, max(confirmations) + 2),
        color="tab:green",
        align="left",
        rwidth=0.8,
    )
    axes[2].set_title("confirmaciones entre niveles (n_levels)")
    axes[2].set_xlabel("niveles de zoom que confirman")
    plt.tight_layout()
    plt.show()

    # --- Resumen visual: detecciones sobre keyframes de muestra ---
    positions = sorted({int(round(v)) for v in np.linspace(0, len(dino_keyframes) - 1, N_OVERVIEW_FRAMES)})
    fig, axes = plt.subplots(1, len(positions), figsize=(6.5 * len(positions), 4.2), squeeze=False)
    for ax, position in zip(axes[0], positions):
        k = dino_keyframes[position]
        frame = load_frame_rgb(str(DINO_VIDEO), k)
        height, width = frame.shape[:2]
        ax.imshow(frame)
        for det in dino_detections_cache.get(k, []):
            ax.scatter(
                [det["x"] * width],
                [det["y"] * height],
                s=70,
                facecolors="none",
                edgecolors="cyan",
                linewidths=1.8,
            )
            box = det["bbox"]
            ax.add_patch(
                plt.Rectangle(
                    (box[0] * width, box[1] * height),
                    (box[2] - box[0]) * width,
                    (box[3] - box[1]) * height,
                    fill=False,
                    edgecolor="cyan",
                    linewidth=0.8,
                )
            )
        ax.set_title(f"keyframe {k}: {len(dino_detections_cache.get(k, []))} detecciones")
        ax.axis("off")
    plt.tight_layout()
    plt.show()
    print(f"total: {len(all_dets)} candidatos en {len(dino_keyframes)} keyframes")
else:
    print("Sin detecciones DINO en caché: no hay resumen que graficar.")

## 4. SAM3 — de los puntos DINO a los masklets

SAM3 rastrea cada punto como un objeto (`obj_id`) con
`start_session → add_prompt → propagate_in_video → close_session`. El proxy de
la sesión es el video de trabajo reescalado a 1024x576 a 15 fps: las máscaras
por frame a 4K no caben en 24 GB. Como los puntos DINO son relativos `[0, 1]`,
sirven tal cual en el proxy.

Modos (`SAM_MODE`):

- **`chunked`** (recomendado): divide el proxy en chunks de `CHUNK_FRAMES` frames
  con `OVERLAP_FRAMES` de solape, abre una sesión por chunk y arrastra cada
  objeto vivo al chunk siguiente con `carry_over_points` (mismo `obj_id` global).
  La VRAM queda acotada por el largo del chunk.
- **`incremental`**: una sola sesión; en cada keyframe se añaden los candidatos
  que no están ya cubiertos por una máscara rastreada (se rellena hacia atrás y
  luego hacia delante).
- **`simple`**: prompt(s) en un solo frame y propagación completa; es el único
  modo que admite `PROMPT_MODE="text"` (SAM3 no permite mezclar texto y puntos en
  la misma llamada).

Filtros de candidatos: `MIN_SCORE`, `MIN_SOLIDITY`, `MIN_AREA`, `MASK_MARGIN_PX`
y `DEDUPE_OVERLAP` (ver `select_new_points` en `sam3_utils`). `MAX_OBJECTS` es el
tope global de masklets.

La figura de revisión por keyframe dibuja: máscaras rastreadas (cian), todos los
candidatos DINO (rojo) y los prompts de este keyframe (verde; amarillo relleno =
objetos arrastrados entre chunks).

In [ ]:
if RUN_SAM:
    from sam3.model_builder import build_sam3_video_predictor
    from sam3.visualization_utils import save_masklet_video

    from fish_segmentation.sam3_utils import (
        add_point_prompt,
        add_text_prompt,
        carry_over_points,
        merge_chunk_outputs,
        object_mask_area,
        plan_chunks,
        propagate_in_video,
        select_new_points,
    )

    proxy_video = OUTPUTS / f"{working_video.stem}_1024x576.mp4"
    if not proxy_video.exists():
        process_video(
            input_path=str(working_video),
            output_path=str(proxy_video),
            target_width=PROXY_WIDTH,
            target_height=PROXY_HEIGHT,
            fps=SESSION_FPS,
        )
    capture = cv2.VideoCapture(str(proxy_video))
    n_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    capture.release()
    print(f"proxy SAM3: {proxy_video.relative_to(ROOT)} ({n_frames} frames @ {SESSION_FPS} fps)")
    if n_frames != n_work_frames:
        print(
            f"AVISO: el proxy tiene {n_frames} frames y el video de trabajo "
            f"{n_work_frames}; los keyframes se leen del video de trabajo y el "
            "proxy se recorre hasta su propio último frame."
        )

    predictor = build_sam3_video_predictor(gpus_to_use=range(torch.cuda.device_count()))
    print("predictor SAM3 construido")

    sam_outputs = {}          # frame global -> outputs de SAM3
    sam_track_log = []        # un registro por obj_id (descubierto o arrastrado)
    sam_keyframe_log = []     # diagnóstico por keyframe del modo incremental/chunked
    sam_chunk_summary = []
    sam_session_id = None
    sam_n_candidates = 0
    sam_n_carried = 0
    sam_next_id = 0
    sam_video_path = None
    sam_tracks_path = None
else:
    proxy_video = None
    n_frames = 0
    predictor = None
    sam_outputs, sam_track_log, sam_keyframe_log, sam_chunk_summary = {}, [], [], []
    sam_session_id, sam_n_candidates, sam_n_carried, sam_next_id = None, 0, 0, 0
    sam_video_path, sam_tracks_path = None, None
    print("RUN_SAM=False: se omiten las etapas de SAM3.")

In [ ]:
# --- Modo simple: prompt(s) en un solo frame + propagación completa ---
if RUN_SAM and SAM_MODE == "simple":
    response = predictor.handle_request(
        request=dict(
            type="start_session",
            resource_path=str(proxy_video),
            offload_state_to_cpu=OFFLOAD_STATE_TO_CPU,
            offload_video_to_cpu=OFFLOAD_VIDEO_TO_CPU,
        )
    )
    sam_session_id = response["session_id"]

    prompt_frame = 0
    prompt_outputs = None
    if PROMPT_MODE == "text":
        prompt_outputs = add_text_prompt(
            predictor, sam_session_id, frame_index=prompt_frame, text=TEXT_PROMPT
        )
        print(f"prompt de texto '{TEXT_PROMPT}' en el frame {prompt_frame}")
    else:
        prompt_candidates = select_new_points(
            dino_detections_cache.get(prompt_frame, []),
            min_score=MIN_SCORE,
            min_solidity=MIN_SOLIDITY,
            min_area=MIN_AREA,
            dedupe_overlap=DEDUPE_OVERLAP,
        )[:MAX_OBJECTS]
        for index, det in enumerate(prompt_candidates):
            prompt_outputs = add_point_prompt(
                predictor,
                sam_session_id,
                frame_index=prompt_frame,
                points=[[det["x"], det["y"]]],
                obj_id=index,
            )
            sam_track_log.append(
                {
                    "obj_id": index,
                    "keyframe": prompt_frame,
                    "x": det["x"],
                    "y": det["y"],
                    "radius": det["radius"],
                    "level": det["level"],
                    "n_levels": det["n_levels"],
                    "score_mean": det["score_mean"],
                    "solidity": det["solidity"],
                }
            )
        sam_next_id = len(prompt_candidates)
        sam_n_candidates = len(dino_detections_cache.get(prompt_frame, []))
        print(f"{sam_next_id} point prompts DINO en el frame {prompt_frame}")

    if prompt_outputs is not None:
        sam_outputs = {prompt_frame: prompt_outputs}
    sam_outputs.update(
        propagate_in_video(predictor, sam_session_id, start_frame_index=prompt_frame)
    )
    print(f"{len(sam_outputs)} frames con salida de SAM3")

In [ ]:
# --- Modo incremental: una sola sesión, DINO en cada keyframe ---
if RUN_SAM and SAM_MODE == "incremental":
    assert PROMPT_MODE == "dino_points", (
        "el modo incremental usa puntos DINO; cambia PROMPT_MODE a 'dino_points' "
        "o usa SAM_MODE='simple' para el prompt de texto"
    )
    response = predictor.handle_request(
        request=dict(
            type="start_session",
            resource_path=str(proxy_video),
            offload_state_to_cpu=OFFLOAD_STATE_TO_CPU,
            offload_video_to_cpu=OFFLOAD_VIDEO_TO_CPU,
        )
    )
    sam_session_id = response["session_id"]
    outputs_per_frame = {}
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()

    for k in dino_keyframes:
        if k >= n_frames:
            break
        detections = dino_detections_cache.get(k, [])
        new_points = select_new_points(
            detections,
            frame_outputs=outputs_per_frame.get(k),
            min_score=MIN_SCORE,
            min_solidity=MIN_SOLIDITY,
            min_area=MIN_AREA,
            mask_margin_px=MASK_MARGIN_PX,
            dedupe_overlap=DEDUPE_OVERLAP,
        )
        accepted = new_points[: max(0, MAX_OBJECTS - sam_next_id)]
        for index, det in enumerate(accepted):
            add_point_prompt(
                predictor,
                sam_session_id,
                frame_index=k,
                points=[[det["x"], det["y"]]],
                obj_id=sam_next_id + index,
            )
            sam_track_log.append(
                {
                    "obj_id": sam_next_id + index,
                    "keyframe": k,
                    "x": det["x"],
                    "y": det["y"],
                    "radius": det["radius"],
                    "level": det["level"],
                    "n_levels": det["n_levels"],
                    "score_mean": det["score_mean"],
                    "solidity": det["solidity"],
                }
            )
        sam_next_id += len(accepted)
        sam_n_candidates += len(detections)
        sam_keyframe_log.append(
            {
                "mode": "incremental",
                "keyframe": k,
                "candidates": len(detections),
                "new": len(new_points),
                "accepted": len(accepted),
                "tracked": sam_next_id,
            }
        )

        if accepted:
            if k > 0:
                outputs_per_frame.update(
                    propagate_in_video(
                        predictor,
                        sam_session_id,
                        start_frame_index=k,
                        propagation_direction="backward",
                    )
                )
            outputs_per_frame.update(
                propagate_in_video(
                    predictor,
                    sam_session_id,
                    start_frame_index=k,
                    propagation_direction="forward",
                )
            )

        if device == "cuda":
            free_bytes, _ = torch.cuda.mem_get_info()
            print(
                f"k={k:5d}: {len(detections):2d} candidatos DINO, "
                f"{len(new_points):2d} nuevos, {len(accepted):2d} prompteados, "
                f"{sam_next_id:2d} masklets | "
                f"VRAM {torch.cuda.memory_allocated() / 2**30:5.2f} GiB alloc, "
                f"{free_bytes / 2**30:5.2f} GiB libre"
            )
        else:
            print(f"k={k:5d}: {len(detections):2d} candidatos DINO, {len(accepted):2d} prompteados")

        # Revisión visual: cian = máscaras rastreadas, rojo = candidatos, verde = prompteados
        if SHOW_KEYFRAME_REVIEW:
            review = cv2.resize(keyframe, (PROXY_WIDTH, PROXY_HEIGHT))
            frame_out = outputs_per_frame.get(k)
            if frame_out is not None:
                for mask in frame_out["out_binary_masks"]:
                    contours, _ = cv2.findContours(
                        mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
                    )
                    cv2.drawContours(review, contours, -1, (0, 255, 255), 2)
            for det in detections:
                center = (int(det["x"] * review.shape[1]), int(det["y"] * review.shape[0]))
                cv2.circle(review, center, 6, (255, 0, 0), 2)
            for det in accepted:
                center = (int(det["x"] * review.shape[1]), int(det["y"] * review.shape[0]))
                cv2.circle(review, center, 6, (0, 255, 0), -1)
            fig, ax = plt.subplots(1, 1, figsize=(12, 7))
            ax.imshow(review)
            ax.set_title(
                f"keyframe {k} | cian: rastreadas, rojo: candidatos, verde: prompteadas"
            )
            ax.axis("off")
            plt.show()

    sam_outputs = outputs_per_frame
    print(
        f"incremental: {len(sam_track_log)} tracks desde {sam_n_candidates} candidatos "
        f"en {len(dino_keyframes)} keyframes; {len(sam_outputs)}/{n_frames} frames con salida"
    )

In [ ]:
# --- Modo chunked: una sesión por chunk, ids globales arrastrados por el solape ---
if RUN_SAM and SAM_MODE == "chunked":
    assert PROMPT_MODE == "dino_points", (
        "el modo chunked usa puntos DINO; cambia PROMPT_MODE a 'dino_points' "
        "o usa SAM_MODE='simple' para el prompt de texto"
    )
    chunk_plan = plan_chunks(n_frames, CHUNK_FRAMES, OVERLAP_FRAMES, align=KEYFRAME_STRIDE)
    print(f"plan de chunks ({CHUNK_FRAMES} frames, {OVERLAP_FRAMES} de solape): {chunk_plan}")

    chunk_dir = OUTPUTS / "chunks"
    chunk_dir.mkdir(parents=True, exist_ok=True)
    chunk_paths = []
    for index, (start, end) in enumerate(chunk_plan):
        chunk_path = chunk_dir / f"{proxy_video.stem}_c{index}_{start}_{end}.mp4"
        if not chunk_path.exists():
            process_video(
                input_path=str(proxy_video),
                output_path=str(chunk_path),
                target_width=PROXY_WIDTH,
                target_height=PROXY_HEIGHT,
                start_time=start / SESSION_FPS,
                duration=(end - start) / SESSION_FPS,
                fps=SESSION_FPS,
            )
        capture = cv2.VideoCapture(str(chunk_path))
        chunk_len_file = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
        capture.release()
        assert chunk_len_file == end - start, (
            f"chunk {index}: {chunk_len_file} frames, se esperaban {end - start}"
        )
        chunk_paths.append(chunk_path)
    print(f"{len(chunk_paths)} proxies de chunk listos en {chunk_dir.relative_to(ROOT)}")

    chunk_frame_outputs = []
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()

    for chunk_index, ((start, end), chunk_path) in enumerate(zip(chunk_plan, chunk_paths)):
        chunk_len = end - start
        response = predictor.handle_request(
            request=dict(
                type="start_session",
                resource_path=str(chunk_path),
                offload_state_to_cpu=OFFLOAD_STATE_TO_CPU,
                offload_video_to_cpu=OFFLOAD_VIDEO_TO_CPU,
            )
        )
        sam_session_id = response["session_id"]
        outputs_per_frame = {}
        carried = []

        # Arrastre: el frame de traspaso es el `start` global de este chunk, que
        # el chunk anterior cubrió (allí su clave local es start - previous_start).
        if chunk_index > 0:
            previous_start, _ = chunk_plan[chunk_index - 1]
            for point in carry_over_points(
                chunk_frame_outputs[-1].get(start - previous_start), min_area=CARRY_MIN_AREA
            ):
                prompt_outputs = add_point_prompt(
                    predictor,
                    sam_session_id,
                    frame_index=0,
                    points=[[point["x"], point["y"]]],
                    obj_id=point["obj_id"],
                )
                if object_mask_area(prompt_outputs, point["obj_id"]) == 0:
                    print(f"chunk {chunk_index}: el objeto arrastrado {point['obj_id']} quedó sin máscara, se descarta")
                    continue
                outputs_per_frame[0] = prompt_outputs  # dedupe del k=0 contra las máscaras arrastradas
                carried.append(point)
                sam_track_log.append(
                    {
                        "obj_id": point["obj_id"],
                        "chunk": chunk_index,
                        "carried_from": chunk_index - 1,
                        "keyframe": start,
                        "x": point["x"],
                        "y": point["y"],
                        "area": point["area"],
                    }
                )
        sam_n_carried += len(carried)

        for k in range(0, chunk_len, KEYFRAME_STRIDE):
            global_k = start + k
            keyframe = load_frame_rgb(str(DINO_VIDEO), global_k)
            if keyframe is None:
                print(f"chunk {chunk_index} k={k:4d} (global {global_k}): frame no legible, se omite")
                continue

            detections = dino_detections_cache.get(global_k, [])
            new_points = select_new_points(
                detections,
                frame_outputs=outputs_per_frame.get(k),
                min_score=MIN_SCORE,
                min_solidity=MIN_SOLIDITY,
                min_area=MIN_AREA,
                mask_margin_px=MASK_MARGIN_PX,
                dedupe_overlap=DEDUPE_OVERLAP,
            )
            accepted = new_points[: max(0, MAX_OBJECTS - sam_next_id)]
            for index, det in enumerate(accepted):
                add_point_prompt(
                    predictor,
                    sam_session_id,
                    frame_index=k,
                    points=[[det["x"], det["y"]]],
                    obj_id=sam_next_id + index,
                )
                sam_track_log.append(
                    {
                        "obj_id": sam_next_id + index,
                        "chunk": chunk_index,
                        "keyframe": global_k,
                        "x": det["x"],
                        "y": det["y"],
                        "radius": det["radius"],
                        "level": det["level"],
                        "n_levels": det["n_levels"],
                        "score_mean": det["score_mean"],
                        "solidity": det["solidity"],
                    }
                )
            sam_next_id += len(accepted)
            sam_n_candidates += len(detections)
            sam_keyframe_log.append(
                {
                    "mode": "chunked",
                    "chunk": chunk_index,
                    "keyframe": global_k,
                    "candidates": len(detections),
                    "new": len(new_points),
                    "accepted": len(accepted),
                    "tracked": sam_next_id,
                }
            )

            # Se propaga si hay algo que rastrear: candidatos nuevos y/o los
            # objetos arrastrados (estos solo en k=0, que si no no llegarían al final).
            if accepted or (k == 0 and carried):
                if k > 0:
                    outputs_per_frame.update(
                        propagate_in_video(
                            predictor,
                            sam_session_id,
                            start_frame_index=k,
                            propagation_direction="backward",
                        )
                    )
                outputs_per_frame.update(
                    propagate_in_video(
                        predictor,
                        sam_session_id,
                        start_frame_index=k,
                        propagation_direction="forward",
                    )
                )

            if device == "cuda":
                free_bytes, _ = torch.cuda.mem_get_info()
                print(
                    f"c{chunk_index} k={k:4d} (global {global_k:4d}): {len(detections):2d} candidatos, "
                    f"{len(new_points):2d} nuevos, {len(accepted):2d} prompteados, {sam_next_id:2d} masklets, "
                    f"{len(outputs_per_frame):3d}/{chunk_len} frames | "
                    f"VRAM {torch.cuda.memory_allocated() / 2**30:5.2f} GiB alloc, "
                    f"{free_bytes / 2**30:5.2f} GiB libre"
                )
            else:
                print(
                    f"c{chunk_index} k={k:4d} (global {global_k:4d}): "
                    f"{len(detections):2d} candidatos, {len(accepted):2d} prompteados"
                )

            if SHOW_KEYFRAME_REVIEW:
                review = cv2.resize(keyframe, (PROXY_WIDTH, PROXY_HEIGHT))
                frame_out = outputs_per_frame.get(k)
                if frame_out is not None:
                    for mask in frame_out["out_binary_masks"]:
                        contours, _ = cv2.findContours(
                            mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
                        )
                        cv2.drawContours(review, contours, -1, (0, 255, 255), 2)
                for det in detections:
                    center = (int(det["x"] * review.shape[1]), int(det["y"] * review.shape[0]))
                    cv2.circle(review, center, 6, (255, 0, 0), 2)
                for det in accepted:
                    center = (int(det["x"] * review.shape[1]), int(det["y"] * review.shape[0]))
                    cv2.circle(review, center, 6, (0, 255, 0), -1)
                if k == 0:
                    for point in carried:
                        center = (
                            int(point["x"] * review.shape[1]),
                            int(point["y"] * review.shape[0]),
                        )
                        cv2.circle(review, center, 6, (0, 255, 255), -1)
                fig, ax = plt.subplots(1, 1, figsize=(12, 7))
                ax.imshow(review)
                ax.set_title(
                    f"chunk {chunk_index} k={k} (global {global_k}) | cian: rastreadas/arrastradas, "
                    "rojo: candidatos, verde: prompteadas"
                )
                ax.axis("off")
                plt.show()

        chunk_frame_outputs.append(outputs_per_frame)
        sam_chunk_summary.append(
            {
                "index": chunk_index,
                "start": start,
                "end": end,
                "carried": len(carried),
                "frames_covered": len(outputs_per_frame),
                "masklets_after": sam_next_id,
            }
        )
        print(
            f"chunk {chunk_index} listo: {len(carried)} arrastrados, "
            f"{len(outputs_per_frame)}/{chunk_len} frames cubiertos, {sam_next_id} masklets"
        )
        predictor.handle_request(request=dict(type="close_session", session_id=sam_session_id))
        sam_session_id = None
        if device == "cuda":
            torch.cuda.empty_cache()

    sam_outputs = merge_chunk_outputs(chunk_frame_outputs, chunk_plan)
    print(
        f"chunked: {len(sam_track_log)} tracks ({sam_n_carried} arrastrados) desde "
        f"{sam_n_candidates} candidatos en {len(chunk_plan)} chunks; "
        f"merge {len(sam_outputs)}/{n_frames} frames"
    )

## 5. Exportación y verificación de tracks

`save_masklet_video` renderiza el video overlay con las máscaras de todos los
frames y los JSON guardan el origen de cada `obj_id` (keyframe de descubrimiento
o arrastre entre chunks). Los plots resumen la salud del tracking: objetos
visibles por frame, matriz de área por objeto (filas = `obj_id`, columnas =
frame; los huecos son frames sin máscara) y el checklist PASS/FALLO final.

In [ ]:
if RUN_SAM and sam_outputs:
    sam_stem = f"pipeline_e2e_{working_video.stem}_{SAM_MODE}_{PROMPT_MODE}"
    sam_video_path = OUTPUTS / f"{sam_stem}.mp4"
    sam_tracks_path = OUTPUTS / f"{sam_stem}_tracks.json"
    sam_summary_path = OUTPUTS / f"{sam_stem}_summary.json"

    proxy_frames = load_all_frames_rgb(str(proxy_video))
    save_masklet_video(
        video_frames=proxy_frames,
        outputs=sam_outputs,
        out_path=str(sam_video_path),
        alpha=0.5,
        fps=SESSION_FPS,
    )
    with open(sam_tracks_path, "w", encoding="utf-8") as handle:
        json.dump(sam_track_log, handle, indent=2)
    with open(sam_summary_path, "w", encoding="utf-8") as handle:
        json.dump(
            {
                "dataset_id": working_video.stem,
                "sam_mode": SAM_MODE,
                "prompt_mode": PROMPT_MODE,
                "text_prompt": TEXT_PROMPT if PROMPT_MODE == "text" else None,
                "session_fps": SESSION_FPS,
                "keyframe_stride": KEYFRAME_STRIDE,
                "max_objects": MAX_OBJECTS,
                "dino_source": DINO_SOURCE,
                "chunk_frames": CHUNK_FRAMES if SAM_MODE == "chunked" else None,
                "overlap_frames": OVERLAP_FRAMES if SAM_MODE == "chunked" else None,
                "n_frames": n_frames,
                "tracks": len(sam_track_log),
                "carried": sam_n_carried,
                "candidates": sam_n_candidates,
                "keyframes": sam_keyframe_log,
                "chunks": sam_chunk_summary,
            },
            handle,
            indent=2,
        )
    print(sam_video_path.relative_to(ROOT))
    print(sam_tracks_path.relative_to(ROOT))
    print(sam_summary_path.relative_to(ROOT))
else:
    print("RUN_SAM=False o sin salidas de SAM3: no hay video que exportar.")

In [ ]:
from IPython.display import Video

if RUN_SAM and sam_video_path is not None and sam_video_path.exists():
    # src relativo para que JupyterLab lo sirva desde notebooks/outputs sin incrustarlo
    Video(f"outputs/{sam_video_path.name}", embed=False, width=640)
else:
    print("Sin video overlay todavía.")

In [ ]:
# --- Estadísticas de los tracks: objetos por frame y área por objeto ---
if RUN_SAM and sam_outputs:
    frame_indices = sorted(sam_outputs)
    tracked_ids = set()
    for outputs in sam_outputs.values():
        if not outputs:
            continue
        for obj_id in np.asarray(outputs.get("out_obj_ids", [])).reshape(-1):
            tracked_ids.add(int(obj_id))
    tracked_ids = sorted(tracked_ids)
    id_to_row = {obj_id: row for row, obj_id in enumerate(tracked_ids)}

    area_matrix = np.zeros((max(1, len(tracked_ids)), len(frame_indices)), dtype=np.float32)
    objects_per_frame = []
    for column, frame_index in enumerate(frame_indices):
        outputs = sam_outputs[frame_index]
        visible = 0
        if outputs:
            masks = np.asarray(outputs.get("out_binary_masks"))
            obj_ids = np.asarray(outputs.get("out_obj_ids", [])).reshape(-1)
            for obj_id, mask in zip(obj_ids, masks):
                area = int(np.count_nonzero(mask))
                if area > 0:
                    visible += 1
                    area_matrix[id_to_row[int(obj_id)], column] = area
        objects_per_frame.append(visible)
    objects_per_frame = np.asarray(objects_per_frame)

    covered_frames = int(np.count_nonzero(objects_per_frame))
    print(f"masklets: {len(tracked_ids)} | frames con máscara: {covered_frames}/{len(frame_indices)} "
          f"({100 * covered_frames / max(1, len(frame_indices)):.1f}%)")
    print(f"objetos visibles por frame: media {objects_per_frame.mean():.2f}, "
          f"máximo {objects_per_frame.max()}")
    for obj_id in tracked_ids:
        row = area_matrix[id_to_row[obj_id]]
        visible = int(np.count_nonzero(row))
        print(f"  obj {obj_id:2d}: visible en {visible:3d}/{len(frame_indices)} frames, "
              f"área mediana {int(np.median(row[row > 0])) if visible else 0} px")

    fig, axes = plt.subplots(1, 2, figsize=(18, 5))
    axes[0].plot(frame_indices, objects_per_frame, color="tab:blue")
    axes[0].set_title("objetos con máscara por frame")
    axes[0].set_xlabel("frame")
    axes[0].set_ylabel("nº objetos")
    axes[0].grid(alpha=0.3)
    image = axes[1].imshow(area_matrix, aspect="auto", interpolation="nearest", cmap="viridis")
    axes[1].set_title("área de máscara por obj_id (filas) y frame (columnas)")
    axes[1].set_xlabel("frame")
    axes[1].set_ylabel("obj_id")
    axes[1].set_yticks(range(len(tracked_ids)))
    axes[1].set_yticklabels(tracked_ids)
    fig.colorbar(image, ax=axes[1], fraction=0.046, pad=0.04, label="píxeles")
    plt.tight_layout()
    plt.show()
else:
    print("Sin salidas de SAM3: no hay estadísticas de tracks.")

In [ ]:
# --- Checklist final: PASS/FALLO por etapa y resumen cuantitativo ---
if device == "cuda":
    peak_gib = torch.cuda.max_memory_allocated() / 2**30
else:
    peak_gib = 0.0

total_candidates = sum(len(dets) for dets in dino_detections_cache.values())
frames_with_masks = 0
if sam_outputs:
    for outputs in sam_outputs.values():
        if not outputs:
            continue
        masks = np.asarray(outputs.get("out_binary_masks"))
        if masks.ndim == 3 and any(int(np.count_nonzero(mask)) > 0 for mask in masks):
            frames_with_masks += 1

checks = {
    "video de trabajo de dataset/processed": PROCESSED_VIDEO.is_file() and n_work_frames > 0,
    "recorte de trabajo en outputs": working_video.is_file(),
    "DINO ejecutado": bool(dino_detections_cache),
    "DINO con candidatos": total_candidates > 0,
    "SAM3 ejecutado": (not RUN_SAM) or bool(sam_outputs),
    "SAM3 con máscaras": (not RUN_SAM) or frames_with_masks > 0,
    "video overlay exportado": sam_video_path is not None and sam_video_path.exists(),
    "tracks JSON exportado": sam_tracks_path is not None and sam_tracks_path.exists(),
}
print("=" * 68)
print("VERIFICACIÓN FINAL DEL PIPELINE (INGESTA -> DINO -> SAM3)")
print("=" * 68)
for name, passed in checks.items():
    print(f"[{'OK   ' if passed else 'FALLO'}] {name}")
print("-" * 68)
print(f"video de trabajo : {working_video.name} ({n_work_frames} frames @ {work_fps:.1f} fps)")
print(f"keyframes DINO   : {len(dino_keyframes)}")
print(f"candidatos DINO  : {total_candidates}")
print(f"tracks SAM3      : {len(sam_track_log)} ({sam_n_carried} arrastrados)")
print(f"frames cubiertos : {frames_with_masks}/{n_frames}")
print(f"VRAM pico        : {peak_gib:.2f} GiB")
print(f"modo             : {SAM_MODE} / {PROMPT_MODE}")

fig, ax = plt.subplots(figsize=(9, 3.2))
labels = ["keyframes DINO", "candidatos DINO", "tracks SAM3", "frames con máscara"]
values = [len(dino_keyframes), total_candidates, len(sam_track_log), frames_with_masks]
ax.barh(labels, values, color=["tab:blue", "tab:orange", "tab:green", "tab:red"])
for row, value in enumerate(values):
    ax.text(value, row, f" {value}", va="center")
ax.set_title("Resumen cuantitativo del pipeline")
plt.tight_layout()
plt.show()

In [ ]:
# --- Limpieza: cerrar la sesión SAM3 y liberar el grupo de procesos ---
if RUN_SAM and predictor is not None:
    if sam_session_id is not None:
        _ = predictor.handle_request(
            request=dict(type="close_session", session_id=sam_session_id)
        )
        sam_session_id = None
        print("sesión SAM3 cerrada")
    predictor.shutdown()
    print("predictor SAM3 finalizado")
else:
    print("SAM3 no se ejecutó en esta corrida.")

!nvidia-smi

## Limitaciones y notas

- DINO no distingue delfines de falsos positivos (glints, olas): cada candidato
  aceptado se convierte en un track. Para contrastar, corre
  `SAM_MODE="simple"` + `PROMPT_MODE="text"` con `TEXT_PROMPT="dolphin"`.
- Las máscaras se calculan en el proxy 1024x576 (no en el 4K) por VRAM; los
  puntos son relativos, así que el prompt no pierde precisión, pero el borde de
  la máscara sí es de proxy.
- En `incremental`/`chunked`, un objeto descubierto a mitad de video recibe
  máscaras *backfill* hacia atrás que pueden ser espurias (SAM3 no expone score
  por frame). En `chunked` los `obj_id` son globales porque cada sesión reusa el
  id arrastrado; un delfín que desaparece justo en un borde de chunk puede
  partirse en dos ids (el solape lo hace improbable, se ve en el JSON).
- `PROMPT_MODE="text"` solo funciona con `SAM_MODE="simple"`; SAM3 no permite
  mezclar texto y puntos en la misma llamada.
- `CHUNK_FRAMES - OVERLAP_FRAMES` debe ser múltiplo de `KEYFRAME_STRIDE` para
  que la grilla global de keyframes no cambie entre chunks (`plan_chunks` lo
  valida).
- Este notebook reutiliza la caché DINO (`<stem>_dino_detections.json`): con
  `RUN_DINO=False` y una caché previa se puede correr solo SAM3.